# Redes Neuronales III – Práctica 1: Técnicas de optimización (Semana 5)

**Caso de uso (sector salud):** clasificación de tumores de mama como benignos o malignos a partir de 30 características de núcleos celulares (dataset *Breast Cancer Wisconsin – Diagnostic*, 569 muestras). Uso exclusivamente académico: no sustituye un diagnóstico médico.

Temas: hiperparámetros, train/dev/test, Feature Scaling, Batch Normalization, mini-batch, Momentum, RMSProp, Adam, Learning Rate Decay.

## 1. Datos: distribución y preprocesamiento

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42

# 1. Carga: Breast Cancer Wisconsin (Diagnostic), incluido en scikit-learn
data = load_breast_cancer(as_frame=True)
X = data.data
y = (1 - data.target).astype("int32")      # 1 = maligno, 0 = benigno

# 2. Distribucion estratificada 60 / 20 / 20 (train / dev / test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED)

# 3. Preprocesamiento: estandarizacion ajustada SOLO con train (evita fuga de datos)
scaler = StandardScaler().fit(X_train)
Xtr_s, Xdv_s, Xte_s = [scaler.transform(a).astype("float32")
                       for a in (X_train, X_dev, X_test)]
Xtr_raw = X_train.to_numpy(dtype="float32")     # sin escalar (solo para comparar)
Xdv_raw = X_dev.to_numpy(dtype="float32")
N_FEATURES = Xtr_s.shape[1]

resumen = pd.DataFrame({
    "conjunto": ["train", "dev", "test"],
    "muestras": [len(y_train), len(y_dev), len(y_test)],
    "malignos": [int(y_train.sum()), int(y_dev.sum()), int(y_test.sum())]})
resumen["% malignos"] = (100 * resumen["malignos"] / resumen["muestras"]).round(1)
print(resumen.to_string(index=False))
print("Caracteristicas:", N_FEATURES)

## 2. Modelo y utilidades (TensorFlow + Keras)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

keras.utils.set_random_seed(SEED)

def build_model(units=(64, 32), use_bn=False, dropout=0.0, l1=0.0, l2=0.0, noise=0.0):
    """MLP binario. noise>0 agrega ruido gaussiano (data augmentation tabular)."""
    reg = regularizers.L1L2(l1=l1, l2=l2) if (l1 or l2) else None
    m = keras.Sequential([layers.Input(shape=(N_FEATURES,))])
    if noise:
        m.add(layers.GaussianNoise(noise))
    for u in units:
        m.add(layers.Dense(u, use_bias=not use_bn, kernel_regularizer=reg))
        if use_bn:
            m.add(layers.BatchNormalization())
        m.add(layers.Activation("relu"))
        if dropout:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation="sigmoid"))
    return m

def compile_model(m, optimizer):
    m.compile(optimizer=optimizer, loss="binary_crossentropy",
              metrics=["accuracy",
                       keras.metrics.Recall(name="recall"),
                       keras.metrics.AUC(name="auc")])
    return m

def entrenar(make_model, make_opt, Xtr=None, Xdv=None, epochs=100, batch_size=32,
             callbacks=None, seeds=(1, 2, 3)):
    """Entrena con varias semillas y promedia, porque el dataset es pequeno."""
    Xtr = Xtr_s if Xtr is None else Xtr
    Xdv = Xdv_s if Xdv is None else Xdv
    filas, hist = [], None
    for s in seeds:
        keras.backend.clear_session()
        keras.utils.set_random_seed(s)
        m = compile_model(make_model(), make_opt())
        h = m.fit(Xtr, y_train, validation_data=(Xdv, y_dev), epochs=epochs,
                  batch_size=batch_size, callbacks=callbacks() if callbacks else [],
                  verbose=0)
        tr = m.evaluate(Xtr, y_train, verbose=0, return_dict=True)
        dv = m.evaluate(Xdv, y_dev, verbose=0, return_dict=True)
        filas.append([tr["accuracy"], dv["accuracy"], dv["recall"], dv["auc"],
                      len(h.history["loss"])])
        hist = h.history
    r = np.array(filas)
    resumen = {"acc_train": r[:, 0].mean(), "acc_dev": r[:, 1].mean(),
               "std_dev": r[:, 1].std(), "recall_dev": r[:, 2].mean(),
               "auc_dev": r[:, 3].mean(), "epocas": r[:, 4].mean()}
    return resumen, hist

## 3. A. Feature Scaling

In [ ]:
# A. Efecto del Feature Scaling (mismo modelo, mismo optimizador)
adam = lambda: keras.optimizers.Adam(1e-3)
res_raw, h_raw = entrenar(lambda: build_model(), adam, Xtr=Xtr_raw, Xdv=Xdv_raw)
res_esc, h_esc = entrenar(lambda: build_model(), adam)

print(pd.DataFrame({"Sin escalar": res_raw, "Estandarizado": res_esc}).round(4))

plt.figure(figsize=(7, 4))
plt.plot(h_raw["val_loss"], label="Sin escalar")
plt.plot(h_esc["val_loss"], label="Estandarizado")
plt.xlabel("Epoca"); plt.ylabel("Perdida en dev"); plt.legend(); plt.grid(alpha=.3)
plt.title("Efecto del Feature Scaling"); plt.show()

## 4. B. Comparación de optimizadores

In [ ]:
# B. Comparacion de optimizadores (mini-batch = 32, 100 epocas, 3 semillas)
decay = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-2, decay_steps=100, decay_rate=0.9)

optimizadores = {
    "SGD":             lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD + Momentum":  lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSProp":         lambda: keras.optimizers.RMSprop(learning_rate=1e-3),
    "Adam":            lambda: keras.optimizers.Adam(learning_rate=1e-3),
    "Adam + LR decay": lambda: keras.optimizers.Adam(learning_rate=decay),
}

tabla, curvas = {}, {}
for nombre, make_opt in optimizadores.items():
    tabla[nombre], curvas[nombre] = entrenar(lambda: build_model(), make_opt)
    print(f"{nombre:16s} acc_dev={tabla[nombre]['acc_dev']:.4f}  recall_dev={tabla[nombre]['recall_dev']:.4f}")

print(pd.DataFrame(tabla).T.round(4))

plt.figure(figsize=(8, 5))
for nombre, h in curvas.items():
    plt.plot(h["val_loss"], label=nombre)
plt.xlabel("Epoca"); plt.ylabel("Perdida en dev"); plt.legend(); plt.grid(alpha=.3)
plt.title("Comparacion de optimizadores"); plt.show()

## 5. C. Batch Normalization

In [ ]:
# C. Efecto de Batch Normalization (Adam)
adam = lambda: keras.optimizers.Adam(1e-3)
res_sin, h_sin = entrenar(lambda: build_model(use_bn=False), adam)
res_con, h_con = entrenar(lambda: build_model(use_bn=True), adam)
print(pd.DataFrame({"Sin BatchNorm": res_sin, "Con BatchNorm": res_con}).round(4))

plt.figure(figsize=(7, 4))
plt.plot(h_sin["val_loss"], label="Sin BatchNorm")
plt.plot(h_con["val_loss"], label="Con BatchNorm")
plt.xlabel("Epoca"); plt.ylabel("Perdida en dev"); plt.legend(); plt.grid(alpha=.3)
plt.title("Efecto de Batch Normalization"); plt.show()

## 6. D. Tamaño del mini-batch

In [ ]:
# D. Efecto del tamano del mini-batch (el ultimo equivale a batch gradient descent)
import time
filas = {}
for bs in (8, 32, 128, len(y_train)):
    t0 = time.time()
    r, _ = entrenar(lambda: build_model(), lambda: keras.optimizers.Adam(1e-3),
                    batch_size=bs)
    r["tiempo_s"] = time.time() - t0
    filas[f"batch={bs}"] = r
print(pd.DataFrame(filas).T.round(4))

## 7. Conclusiones

*Escribe aquí qué configuración elegirías y por qué, con base en las tablas.*